# Improve the classifier from the log
### Clinic Appointment Assistant · Capstone (criterion 3.6)

1. Download the log from the Google Sheet (File, Download, Comma separated values). Rename it `log.csv`.
2. Upload `log.csv`, `intents.csv` (the one from Portfolio Project 2 Part 1) to the Files panel on the left.
3. Run the cells from the top. In Step 2, write the messages YOU label, from the rows worth reading.
4. Download the new `intents.csv` and `intent_classifier.joblib`, and upload them to GitHub.

**The log file itself is never uploaded to GitHub.** No names or phone numbers in the examples.

In [ ]:
import re
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

THRESHOLD = 0.2          # the value chosen in Portfolio Project 2 Part 1

def clean(text):
    return re.sub(r"[^\w\s:]", "", str(text)).lower().strip()

df = pd.read_csv("intents.csv")
print(len(df), "rows;", (df["split"] == "test").sum(), "test rows")

## Step 1: The rows worth reading

The fallback, a 👎, an answer with low confidence. Read these first.

In [ ]:
log = pd.read_csv("log.csv")
turns = log[log["message"].notna() & (log["feedback"].isna() | (log["feedback"] == ""))]
to_read = log[(log["answered_by"] == "fallback")
              | (log["feedback"] == "👎")
              | ((log["answered_by"] == "classifier") & (log["confidence"].astype(float) < THRESHOLD + 0.05))]
print(len(turns), "turns in the log;", len(to_read), "worth reading:")
print(to_read[["message", "intent", "confidence", "answered_by", "feedback"]].to_string(index=False))

## Step 2: YOUR work: label them

For each message worth reading that is about the clinic, write the intent it should have had. Messages that are out of scope can be added as `out_of_scope`. Copy each message exactly as it was typed, with no names or numbers.

In [ ]:
# YOUR DOMAIN: (message, the intent it should have had), from the log.
LABELLED = [
    # ("hw mch is teeth cleaning", "prices"),
]
print(len(LABELLED), "messages labelled")

## Step 3: Before and after, on the SAME test set

In [ ]:
def train_model(rows):
    # characters, as chosen in Portfolio Project 2 Part 1
    model = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)),
                          LogisticRegression(max_iter=1000))
    model.fit([clean(t) for t in rows["text"]], list(rows["intent"]))
    return model

def outcomes(model, test):
    right = missed = wrong = 0
    for text, label in zip(test["text"], test["intent"]):
        row = model.predict_proba([clean(text)])[0]
        best = row.argmax()
        guess = model.classes_[best] if row[best] >= THRESHOLD else "out_of_scope"
        if guess == label:
            right += 1
        elif guess == "out_of_scope":
            missed += 1
        else:
            wrong += 1
    return right, missed, wrong

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

before = train_model(train)
b = outcomes(before, test)
print("BEFORE: %d right, %d missed, %d wrong, of %d" % (b + (len(test),)))

new = pd.DataFrame([(t, i, "log", "train") for t, i in LABELLED],
                   columns=["text", "intent", "source", "split"])
df_after = pd.concat([df, new], ignore_index=True)
same = df_after[df_after["split"] == "test"][["text", "intent"]].values.tolist() == test[["text", "intent"]].values.tolist()
assert same, "the test rows changed!"

after = train_model(df_after[df_after["split"] == "train"])
a = outcomes(after, test)
print("AFTER:  %d right, %d missed, %d wrong, of %d" % (a + (len(test),)))
print(len(new), "rows added to training; the", len(test), "test rows are unchanged.")

## Step 4: Keep the new model only if it is better

In [ ]:
if a[0] >= b[0]:
    df_after.to_csv("intents.csv", index=False)
    joblib.dump(after, "intent_classifier.joblib")
    print("New intents.csv and intent_classifier.joblib saved: download them.")
else:
    print("The new model is not better: keep the old one.")